# 090 — Generación musical y de audio

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** 11 bits por índice (log₂ 2048), 44 bits por frame,
2.2 kbps — un factor ~233× frente a PCM. Menos codebooks y menos frames/s que
EnCodec dan un bitrate menor, a costa de calidad de reconstrucción.

**Ejercicio 2.** Aplanar da 36 000 tokens para 60 s (inviable con atención
cuadrática); el delay pattern lo deja en ~4 507 pasos; la onda cruda serían
1 440 000 pasos. Este cálculo ES el argumento a favor de los códecs neuronales.

**Ejercicio 3.** (a) **AR sobre tokens**: genera secuencialmente y puede
continuar un prefijo de audio real. (b) **Difusión sobre mel**: el espectrograma
es una imagen, hereda img2img/inpainting. (c) **Códec truncado**: para comprimir
no hace falta generar; truncar la cascada RVQ baja el bitrate de forma escalable.


In [ ]:
# Ejercicio 1 — bitrate del codec y factor de compresion
import math

sr, fps, Q, K = 32_000, 50, 4, 2048
bits_indice = math.log2(K)
bits_frame = Q * bits_indice
bitrate = fps * bits_frame
pcm = sr * 16
print(f"bits por indice : {bits_indice:.0f}")
print(f"bits por frame  : {bits_frame:.0f}")
print(f"bitrate         : {bitrate:.0f} bits/s = {bitrate/1000:.1f} kbps")
print(f"PCM             : {pcm:,} bits/s = {pcm/1000:.0f} kbps")
print(f"compresion      : {pcm/bitrate:.1f}x")

# referencia EnCodec del ejemplo: 24 kHz, 75 fr/s, 8 codebooks de 1024
ref = 75 * 8 * math.log2(1024)
print(f"referencia EnCodec: {ref:.0f} bits/s = {ref/1000:.0f} kbps")


In [ ]:
# Ejercicio 2 — longitudes de secuencia para 60 s de musica
segundos, fps, Q, sr = 60, 75, 8, 24_000

frames = segundos * fps
aplanado = frames * Q
delay = frames + Q - 1
crudo = segundos * sr
print(f"frames                 : {frames:,}")
print(f"tokens aplanados       : {aplanado:,}")
print(f"pasos con delay pattern: {delay:,}")
print(f"muestras de onda cruda : {crudo:,}")
print(f"crudo / delay          : {crudo/delay:.0f}x mas pasos")


**Ejercicio 4.** El contrato mínimo se valida sin asumir valores internos:


In [ ]:
result = run_lab("generation", seed=90)
assert result["kind"] == "generation"
assert result["evidence"]
show(result)


## Reflexión

1. ¿Por qué la primera capa del RVQ concentra la información "gruesa" de la señal y qué implica truncar la cascada a menos codebooks para el bitrate y la calidad?
2. ¿Qué problema resuelve el delay pattern de MusicGen frente a aplanar los 8 codebooks en una sola secuencia, y cuál es su costo?
3. La difusión sobre espectrogramas mel necesita un vocoder para producir la onda: ¿por qué (qué información falta) y qué artefactos puede introducir esa etapa extra?
